In [ ]:
from itertools import combinations
import numpy as np
import os
import pandas as pd
from PIL import Image
import shutil

import sys
sys.path.append('.')
from config import PROJECT_PATH

This notebook downloads the VinDr-CXR dataset from Kaggle, applies the filtering pipeline and generates the final dataset with images at 1024×1024 and 256×256.

Kaggle API credentials are required to download the dataset. You can either place your `kaggle.json` file in `~/.kaggle/` or uncomment and fill in the credentials directly in the cell below.

In [ ]:
# Set Kaggle API credentials to be able to download the dataset
# os.environ['KAGGLE_USERNAME'] = ""
# os.environ['KAGGLE_KEY'] = ""

In [ ]:
# Dataset xhlulu kaggle (images 1024x1024 PNG)
!kaggle datasets download -d xhlulu/vinbigdata-chest-xray-resized-png-1024x1024
!unzip -q vinbigdata-chest-xray-resized-png-1024x1024.zip -d {PROJECT_PATH}/dataset/
!rm vinbigdata-chest-xray-resized-png-1024x1024.zip

# Official dataset VinDr-CXR (file: train.csv)
!kaggle competitions download -c vinbigdata-chest-xray-abnormalities-detection -f train.csv
!mv train.csv {PROJECT_PATH}/dataset/train.csv 

print("Dataset VinDr-CXR 1024x1024 PNG downloaded and ready for processing.")

In [ ]:
print(f"Total of downloaded images: {len(os.listdir(f'{BASE_PATH}/dataset/train'))}")

## Reescale coordinates and define needed functions


In [ ]:
# Load tags (official) y metadata (original dimensions to reescale coordenates)
df_tags = pd.read_csv(f'{PROJECT_PATH}/dataset/train.csv')
df_meta = pd.read_csv(f'{PROJECT_PATH}/dataset/train_meta.csv') # This one has 'width' and 'height' original dimensions

# We merge using the correct names of the dataset of xhlulu (dim1=width, dim0=height)
df_master = pd.merge(df_tags, df_meta[['image_id', 'dim0', 'dim1']], on='image_id')

# ¿are they already scaled?
if df_master['x_max'].max() > 1024:
    print("Original coordinates detected. Rescaling to 1024x1024...")

    df_master['x_min'] = df_master['x_min'] * (1024 / df_master['dim1'])
    df_master['x_max'] = df_master['x_max'] * (1024 / df_master['dim1'])
    df_master['y_min'] = df_master['y_min'] * (1024 / df_master['dim0'])
    df_master['y_max'] = df_master['y_max'] * (1024 / df_master['dim0'])

    print("Coordinates scaled successfully.")
else:
    print("Coordinates already scaled. No action needed.")

In [ ]:
def calculate_iou(box1, box2):
    """
    box: (x_min, y_min, x_max, y_max)
    """
    x_min = max(box1[0], box2[0])
    y_min = max(box1[1], box2[1])
    x_max = min(box1[2], box2[2])
    y_max = min(box1[3], box2[3])

    intersection = max(0, x_max - x_min) * max(0, y_max - y_min)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

# IoU mínimo por pares entre radiólogos
def minimum_iou_between_pairs(grupo):
    """
    Given a group of rows (annotations from different radiologists on the same image and class), 
    calculate the minimum IoU between all possible pairs.
    If there is only one radiologist with consensus, return 1.0.

    """
    boxes = grupo[['x_min', 'y_min', 'x_max', 'y_max']].values
    if len(boxes) < 2:
        return 1.0
    ious = [calculate_iou(b1, b2) for b1, b2 in combinations(boxes, 2)]
    return min(ious)

## Filter dataset pipeline

In [ ]:
N_HEALTHY = 4000

# Step 0: initial state dataset
print("=" * 60)
print("Step 0 - Initial state of the dataset")
print("=" * 60)
clases_tfg = [0, 3, 14]
df_interest = df_master[df_master['class_id'].isin(clases_tfg)].copy()
print(f"Total rows (annotations): {len(df_interest)}")
print(f"Unique images:             {df_interest['image_id'].nunique()}")
print()

# Step 1: Consensus filtering (≥ 2 different radiologists)
print("=" * 60)
print("Step 1 - Consensus filtering (≥ 2 different radiologists)")
print("=" * 60)

votes = (df_interest
         .groupby(['image_id', 'class_id'])['rad_id']
         .nunique()
         .reset_index()
         .rename(columns={'rad_id': 'n_votes'}))

consensus_ids = votes[votes['n_votes'] >= 2][['image_id', 'class_id']]
df_consensus = pd.merge(df_interest, consensus_ids, on=['image_id', 'class_id'])

# Only pathological classes (aneurysm and cardiomegaly) for the next steps, we will add healthy at the end
df_pathological = df_consensus[df_consensus['class_id'] != 14]

print(f"Images with semantic consensus (pathological): {df_pathological['image_id'].nunique()}")
print(df_pathological.groupby('class_name')['image_id'].nunique().to_string())
print()

# Step 2: Spatial filter (minimum IoU per pair ≥ 0.4)
print("=" * 60)
print("Step 2 — Spatial filter (minimum IoU per pair ≥ 0.4)")
print("=" * 60)

IOU_UMBRAL = 0.4

iou_por_grupo = (df_pathological
                 .groupby(['image_id', 'class_id'])
                 .apply(minimum_iou_between_pairs)
                 .reset_index()
                 .rename(columns={0: 'iou_min'}))

# Imágenes que superan el umbral
aproved = iou_por_grupo[iou_por_grupo['iou_min'] >= IOU_UMBRAL][['image_id', 'class_id']]
discarded_iou = iou_por_grupo[iou_por_grupo['iou_min'] < IOU_UMBRAL]

print(f"Umbral IoU: {IOU_UMBRAL}")
print(f"Images discarded due to low spatial agreement: {len(discarded_iou)}")
print(f"  → Aneurisma discarded:    {len(discarded_iou[discarded_iou['class_id'] == 0])}")
print(f"  → Cardiomegalia discarded:{len(discarded_iou[discarded_iou['class_id'] == 3])}")
print()

df_aproved = pd.merge(df_pathological, aproved, on=['image_id', 'class_id'])
print(f"Images that pass the spatial filter: {df_aproved['image_id'].nunique()}")
print(df_aproved.groupby('class_name')['image_id'].nunique().to_string())
print()

# Step 3: Bounding Box Union (for each image-class, unify into a single BB that encompasses all radiologists)
print("=" * 60)
print("Step 3 — Bounding Box Union")
print("=" * 60)

final_BB = (df_aproved
            .groupby(['image_id', 'class_id', 'class_name'])
            .agg(x_min=('x_min', 'min'),
                 y_min=('y_min', 'min'),
                 x_max=('x_max', 'max'),
                 y_max=('y_max', 'max'))
            .reset_index())

print(f"Final BBs generated: {len(final_BB)}")
print(final_BB.groupby('class_name')['image_id'].nunique().to_string())
print()

# Step 4: Control group (healthy)
print("=" * 60)
print("Step 4 — Control group (healthy)")
print("=" * 60)

# Only images where all radiologists said "no finding" (class_id=14)
ids_healthy = (df_master.groupby('image_id')['class_id']
                  .apply(lambda x: (x == 14).all())
                  .reset_index()
                  .query('class_id == True')['image_id'])

df_healthy = (df_master[df_master['image_id'].isin(ids_healthy)]
            .drop_duplicates('image_id')
            .sample(n=N_HEALTHY, random_state=42))

print(f"Healthy images selected: {len(df_healthy)}")
print()

# Step 5: Final dataset
print("=" * 60)
print("Step 5 — Final dataset")
print("=" * 60)

df_1024 = pd.concat([final_BB, df_healthy[['image_id', 'class_id', 'class_name',
                                               'x_min', 'y_min', 'x_max', 'y_max']]])

# Sequential ID mapping
mapping_ids = {
    0: 0,
    3: 1,
    14: 2
}
#If you need to map the class names to another language
#mapping_names = {
#    'Aortic enlargement': 'aneurisma dell'aorta',
#    'Cardiomegaly':       'cardiomegalia',
#    'No finding':         'nessun risultato'
#}
df_1024['class_id'] = df_1024['class_id'].map(mapping_ids)
#df_1024['class_name'] = df_1024['class_name'].map(mapping_names)

# Images with both pathologies
ids_aneurysm     = set(df_1024[df_1024['class_id'] == 0]['image_id'])
ids_cardiomegaly = set(df_1024[df_1024['class_id'] == 1]['image_id'])
both = ids_aneurysm & ids_cardiomegaly

print(f"Aortic aneurysm:      {len(ids_aneurysm)}")
print(f"Cardiomegaly:  {len(ids_cardiomegaly)}")
print(f"Healthy:          {len(df_healthy)}")
print(f"Both:          {len(both)}")
print(f"Total unique images: {df_1024['image_id'].nunique()}")
print("=" * 60)

## Compress Dataset

In [ ]:
os.makedirs(f'{PROJECT_PATH}/images_1024', exist_ok=True)
os.makedirs(f'{PROJECT_PATH}/images_256', exist_ok=True)

print("Processing images...")

for img_id in df_1024['image_id'].unique():
    src = f'{PROJECT_PATH}/dataset/train/{img_id}.png'
    if os.path.exists(src):
        # Save 1024x1024 version
        shutil.copy(src, f'{PROJECT_PATH}/images_1024/{img_id}.png')

        # Generate and save the 256x256 version (Bicubic)
        img = Image.open(src)
        img_256 = img.resize((256, 256), resample=Image.BICUBIC)
        img_256.save(f'{PROJECT_PATH}/images_256/{img_id}.png')

# Save the metadata within the same folder
df_1024.to_csv(f'{PROJECT_PATH}/metadata_1024.csv', index=False)

# Create the CSV for 256x256, scaling the coordinates (factor 0.25)
df_256 = df_1024.copy()
for col in ['x_min', 'y_min', 'x_max', 'y_max']:
    df_256[col] = df_256[col] * 0.25
df_256.to_csv(f'{PROJECT_PATH}/metadata_256.csv', index=False)

# Clean and compress
os.makedirs(PROJECT_PATH, exist_ok=True)
shutil.rmtree('dataset')
!zip -rq dataset.zip {PROJECT_PATH}

print("Dataset imported!")